# Attention Sinks in Gemma 4 E2B

**Question.** In a *full*-attention layer, the model concentrates a
disproportionate share of its softmax mass on the first one or two
tokens of the sequence — the BOS / first-position *attention sink*.
The leading explanation (Cancedda 2024; Gu et al. 2025) is that this
gives the model a low-information "no-op" target so the residual stream
can pass through the attention block uncorrupted when nothing
particularly relevant is in context.

But Gemma 4 E2B has *sliding-window* attention on 28 of its 35 layers
(window = 512). Once a query position `q` is past the window, **the
sink is gone** — token 0 isn't even visible. So where does the
"leftover" attention mass go?

Hypotheses:

1. **Window-edge sink** — a new sink emerges at the *oldest* token
   still in the window (a positional sink that follows q).
2. **No sink** — sliding layers learn a "no-op" by other means
   (e.g. low-V attention to whichever token happens to be there) and
   the mass spreads roughly uniformly.
3. **Recency** — mass piles up on the last few tokens.
4. **Token-typed sinks** — specific tokens (newlines, punctuation,
   chat-template markers) become preferred destinations regardless
   of position.

We capture per-layer attention from `capture.py` and check.


In [ ]:
%matplotlib inline
from pathlib import Path
import torch, numpy as np
import matplotlib.pyplot as plt

REPO    = Path('/Users/soumil/Code/Projects/Understand/Gemma')
OUT_DIR = REPO / 'experiments' / 'AttentionSinks' / 'outputs'

bundles = []
for p in sorted(OUT_DIR.glob('conv_*.pt')):
    b = torch.load(p, weights_only=False, map_location='cpu')
    bundles.append(b)
    print(f"{p.name}: S={b['input_ids'].shape[1]}, "
          f"layers={len(b['layer_types'])}, window={b['sliding_window']}")
print(f"loaded {len(bundles)} bundle(s)")


## 1. Orientation — what's in a bundle?

`bundle['attentions']` is a list of 35 tensors (one per decoder layer),
each of shape `(num_heads=8, S, S)` in fp16. `attentions[L][h, q, k]` is
the softmax weight that head `h` of layer `L` placed on key `k` while
attending from query `q`. Causal-masked positions are zero. For sliding
layers, positions outside the window are also zero.

`layer_types` tells us which layers are sliding vs full. Pattern in E2B
is 4-sliding : 1-full repeated, with one extra full at the end.


In [ ]:
b = bundles[0]
S = b['input_ids'].shape[1]
W = b['sliding_window']
print(f"S = {S}  (sliding window W = {W})")
print(f"BOS (token 0) leaves the window for queries q >= {W}")
print(f"  queries [0..{W-1}] : sliding can still see BOS")
print(f"  queries [{W}..{S-1}] : sliding has BOS evicted")
print()
print("layer types:")
for i, t in enumerate(b['layer_types']):
    flag = 'S' if t == 'sliding_attention' else 'F'
    print(f"  layer {i:2d}  [{flag}]  {t}")


## 2. Helpers — pull attention rows and slice them up

We'll need three utilities that we'll reuse throughout:

- `attn(b, layer, q)` → `(H, S)` softmax row for one query
- `mass_first_k_in_seq(b, layer, q, k)` — mass on absolute positions 0..k-1 (the *classical* sink)
- `decompose(b, layer, q)` — split mass into `{window-edge, middle, recent}` regions


In [ ]:
def attn(b, layer, q):
    """Return (H, S) softmax row at query q, layer `layer`, in fp32."""
    return b['attentions'][layer][:, q, :].float()


def mass_first_k_in_seq(b, layer, q, k=4):
    """Mass placed on absolute positions 0..k-1.
    For sliding layers with q >= W, this is exactly 0."""
    return attn(b, layer, q)[:, :k].sum(dim=-1).numpy()


def decompose(b, layer, q, edge_k=4, recent_k=32):
    """Split attention[q, :] into three non-overlapping regions:

        edge   = first edge_k positions of the visible window
                  (positions [0..edge_k-1] for full,
                   [q-W+1..q-W+edge_k] for sliding when q >= W-1)
        recent = last recent_k positions ending at q
        middle = everything else (still inside the visible window)

    Returns (edge, middle, recent) each of shape (H,).
    """
    a = attn(b, layer, q).numpy()                    # (H, S)
    H = a.shape[0]
    is_sliding = b['layer_types'][layer] == 'sliding_attention'
    W = b['sliding_window']

    edge_start   = max(0, q - W + 1) if is_sliding else 0
    edge_end     = min(edge_start + edge_k, q + 1)
    recent_start = max(0, q - recent_k + 1)
    recent_end   = q + 1

    edge   = a[:, edge_start:edge_end].sum(axis=-1)
    recent = a[:, recent_start:recent_end].sum(axis=-1)

    overlap_start = max(edge_start, recent_start)
    overlap_end   = min(edge_end, recent_end)
    overlap = (a[:, overlap_start:overlap_end].sum(axis=-1)
               if overlap_end > overlap_start else np.zeros(H))

    total  = a.sum(axis=-1)
    middle = total - edge - recent + overlap
    return edge, middle, recent


## 3. Q1 — does the sink survive sliding?

Plot, per layer, the average mass that the *last* query token places on
absolute positions 0..3. For full layers we expect the classical sink
(non-zero, often large). For sliding layers, this should be exactly 0
because positions 0..3 are masked out of the window.


In [ ]:
b = bundles[0]
S = b['input_ids'].shape[1]
L = len(b['layer_types'])
q = S - 1   # the last (= most-evicted) query

mass_seq04 = np.array([mass_first_k_in_seq(b, l, q, k=4).mean() for l in range(L)])
sliding = np.array([t == 'sliding_attention' for t in b['layer_types']])

fig, ax = plt.subplots(figsize=(11, 3.5))
colors = ['tab:red' if s else 'tab:blue' for s in sliding]
ax.bar(range(L), mass_seq04, color=colors, edgecolor='black', linewidth=0.3)
ax.set_xlabel('layer'); ax.set_ylabel('mass on tokens [0..3]')
ax.set_title(f"Q1 — classical sink mass at q={q} (head-avg per layer)\n"
             f"red = sliding (BOS evicted), blue = full (BOS visible)")
ax.set_xticks(range(L))
plt.tight_layout(); plt.show()


## 4. Q2 — where does the leftover mass go in sliding layers?

For each layer, decompose the same `q = last` row into:

- `edge`   : first 4 positions of the *visible* window
- `recent` : last 32 positions (incl. self)
- `middle` : everything else inside the window

For **full** layers, `edge` is the classical sink (positions 0–3).
For **sliding** layers, `edge` is the *new* candidate sink at the
window's left boundary. If H1 (window-edge sink) is right, sliding
bars should look much like full bars in the `edge` slot.


In [ ]:
edges, mids, recents = [], [], []
for l in range(L):
    e, m, r = decompose(b, l, q)
    edges.append(e.mean()); mids.append(m.mean()); recents.append(r.mean())
edges, mids, recents = map(np.array, (edges, mids, recents))

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(L)
ax.bar(x, edges,                                label='edge (first 4 in window)', color='tab:red')
ax.bar(x, mids,    bottom=edges,                label='middle',                  color='lightgray')
ax.bar(x, recents, bottom=edges + mids,         label='recent (last 32)',        color='tab:green')

# annotate sliding vs full on the x-axis
for i, t in enumerate(b['layer_types']):
    ax.text(i, -0.04, 'S' if t == 'sliding_attention' else 'F',
            ha='center', va='top',
            color='tab:red' if t == 'sliding_attention' else 'tab:blue',
            fontsize=8, fontweight='bold')

ax.set_xlabel('layer'); ax.set_ylabel('attention mass'); ax.set_ylim(0, 1.05)
ax.set_title(f"Q2 — decomposition of attention[q={q}] per layer (head-avg)")
ax.set_xticks(x); ax.legend(loc='upper right', framealpha=0.9)
plt.tight_layout(); plt.show()

print('summary:')
print(f"  sliding-layer mean edge mass   = {edges[sliding].mean():.3f}")
print(f"  sliding-layer mean recent mass = {recents[sliding].mean():.3f}")
print(f"  sliding-layer mean middle mass = {mids[sliding].mean():.3f}")
print(f"  full-layer    mean edge mass   = {edges[~sliding].mean():.3f}")
print(f"  full-layer    mean recent mass = {recents[~sliding].mean():.3f}")


## 5. Q3 — per-head specialisation

A layer-averaged number hides head structure. Some heads might own the
sink and others the recency. Heatmap of `edge` mass at `(layer, head)`.


In [ ]:
H = b['attentions'][0].shape[0]
edge_grid   = np.zeros((L, H))
recent_grid = np.zeros((L, H))
for l in range(L):
    e, _, r = decompose(b, l, q)
    edge_grid[l] = e
    recent_grid[l] = r

fig, axes = plt.subplots(1, 2, figsize=(11, 7))
for ax, grid, name in zip(axes, [edge_grid, recent_grid],
                          ['edge (first 4 in window)', 'recent (last 32)']):
    im = ax.imshow(grid, aspect='auto', cmap='viridis', vmin=0, vmax=1)
    ax.set_xlabel('head'); ax.set_ylabel('layer')
    ax.set_title(name)
    ax.set_yticks(range(L))
    ax.set_yticklabels([f"{i} {'S' if b['layer_types'][i]=='sliding_attention' else 'F'}"
                        for i in range(L)], fontsize=7)
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"Q3 — per-(layer, head) mass at q={q}")
plt.tight_layout(); plt.show()


## 6. Q4 — full attention matrix view

For one sliding head and one full head, plot `log10(attn[q, k])` as a
2-D image. Vertical stripes = a fixed key receives lots of attention
across many queries (a sink). Horizontal bands = a query attends
broadly. The dashed line at `q = W` marks where BOS first leaves the
window for the sliding head.


In [ ]:
sliding_layers = [i for i, t in enumerate(b['layer_types']) if t == 'sliding_attention']
full_layers    = [i for i, t in enumerate(b['layer_types']) if t == 'full_attention']

ls = sliding_layers[len(sliding_layers) // 2]   # a middle sliding layer
lf = full_layers[0]                              # the first full layer

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, layer, name in zip(axes, [ls, lf], ['sliding', 'full']):
    a = b['attentions'][layer].float().mean(dim=0).numpy()         # (S, S), head-avg
    im = ax.imshow(np.log10(a + 1e-8), cmap='magma', vmin=-4, vmax=0,
                   aspect='equal')
    ax.axhline(b['sliding_window'], color='cyan', linestyle='--', linewidth=1,
               label=f"q = window ({b['sliding_window']})")
    ax.axvline(b['sliding_window'], color='cyan', linestyle=':',  linewidth=0.7)
    ax.set_xlabel('key k'); ax.set_ylabel('query q')
    ax.set_title(f"layer {layer} ({name}) — log10 attention, head-avg")
    ax.legend(loc='lower right', fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()


## 7. Q5 — which tokens get picked as sinks?

For sliding layers, take a few late query positions and look at the
top-3 destination keys. Is it always the same kind of token (newline,
chat-template marker, punctuation), or does it drift?


In [ ]:
def topk_dest(b, layer, q, k=3, head_avg=True):
    a = attn(b, layer, q).numpy()
    if head_avg:
        a = a.mean(axis=0, keepdims=True)
    out = []
    for h in range(a.shape[0]):
        top = np.argsort(a[h])[::-1][:k]
        out.append([(int(i), float(a[h, i]), b['tokens'][int(i)]) for i in top])
    return out

print(f"=== conv 0, late queries — top-3 destinations per head-avg sliding layer ===")
for layer in sliding_layers[::4]:        # subsample
    print(f"\nlayer {layer}  (sliding):")
    for qq in [W // 2, W + 32, S - 1]:
        if qq >= S: continue
        top = topk_dest(b, layer, qq, k=3, head_avg=True)[0]
        ctx = '  '.join(f"k={i}({tok!r}, {v:.2f})" for i, v, tok in top)
        print(f"  q={qq:>4d}: {ctx}")


## 8. Q6 — does the sink track query position? (window-edge hypothesis)

If H1 is right, the *argmax* destination of a late sliding-layer query
should sit close to `q - W + 1` (the leftmost visible position) — and
should *move* as `q` grows. Plot argmax-position vs query for a sliding
head, with the `q - W + 1` line overlaid.


In [ ]:
def argmax_track(b, layer, head, q_lo, q_hi):
    A = b['attentions'][layer][head].float().numpy()   # (S, S)
    return np.array([A[q].argmax() for q in range(q_lo, q_hi)])

q_lo, q_hi = max(8, W // 2), S
xs = np.arange(q_lo, q_hi)

fig, ax = plt.subplots(figsize=(11, 4))
for layer in sliding_layers[::6]:
    am = argmax_track(b, layer, head=0, q_lo=q_lo, q_hi=q_hi)
    ax.plot(xs, am, label=f"layer {layer} h0", alpha=0.7, linewidth=1)
ax.plot(xs, np.maximum(0, xs - W + 1), color='black', linestyle='--', label='q − W + 1 (window left edge)')
ax.plot(xs, xs, color='gray', linestyle=':', label='q (self)')
ax.set_xlabel('query position q'); ax.set_ylabel('argmax key')
ax.set_title('Q6 — does the argmax key track the moving window edge?')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


## Findings

> _To be filled in after running. Numbers above will populate the
> story. Initial expectations:_
>
> - **H1 (window-edge sink)**: easiest to see in §4 — if sliding `edge`
>   bars are tall, the model has learned a moving sink at the leftmost
>   visible token.
> - **H2 (no sink, spread mass)**: would show as `middle` dominating
>   in sliding layers and high entropy.
> - **H3 (recency)**: sliding `recent` bar would dominate.
> - **H4 (token-typed sinks)**: §7's argmax tokens would all be the
>   same string (e.g. `'\n'`, `'<start_of_turn>'`).
>
> The most interesting outcome is a *mixture* — some heads pick H1,
> others H3, others H4 — and the model's residual stream survives
> because at least one head per layer is doing the no-op job.
